# Galileo, frozen encoder (variant-dependent embeddings)

This notebook follows `src/models/galileo.py` and shows how the wrapper converts the shared benchmark object into Galileo's grouped space-time inputs.

Key properties:
- Uses the NASA Harvest Galileo v1 wrapper vendored under `src/utils/galileoutil`.
- Supports `nano`, `tiny`, and `base` variants.
- Packs S2, S1, NDVI, climate, location, and time metadata into Galileo input groups.
- Pools the encoder output into one embedding per parcel.

## What Galileo expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `x_st` | `(N, 1, 1, T, 13)` | space-time remote sensing channels |
| `sp_x` | `(N, 1, 1, 16)` | static spatial channels |
| `t_x` | `(N, T, 6)` | time-only metadata |
| `st_x` | `(N, 18)` | static metadata |
| masks | grouped shapes matching inputs | missing-input indicators expected by Galileo |
| output embedding | `(N, D)` | pooled frozen representation, where `D` depends on the variant |

The wrapper handles missing groups by filling values and marking them masked.

In [1]:
import os
import sys
import importlib.util
from pathlib import Path
import numpy as np
from dataio.get_input import Benchmark, ModalitySeries, NativeSeries

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

In [2]:
galileo = load('galileo_mod', 'src/models/galileo.py')

print('variants:', galileo.GALILEO_VARIANTS)
print('space-time bands:', galileo.SPACE_TIME_BANDS)

variants: {'nano': (128, 'models/nano'), 'tiny': (192, 'models/tiny'), 'base': (768, 'models/base')}
space-time bands: ['VV', 'VH', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI']


In [3]:
rng = np.random.default_rng(7)
N, T = 4, 18


s2_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

months = np.arange(T, dtype=np.int64) % 12
doy = np.linspace(15, 350, T).astype(np.float32)
years = np.full(T, 2021, dtype=np.int64)

s2_values = [(rng.random((T, len(s2_bands))) * 5000).astype(np.float32) for _ in range(N)]
s1_values = [rng.normal(-12, 3, size=(T, len(s1_bands))).astype(np.float32) for _ in range(N)]
climate_values = [rng.normal(0, 1, size=(T, len(climate_bands))).astype(np.float32) for _ in range(N)]

native = NativeSeries(
    s2=ModalitySeries(s2_values, [months] * N, [doy] * N, [years] * N, s2_bands),
    s1=ModalitySeries(s1_values, [months] * N, [doy] * N, [years] * N, s1_bands),
    climate=ModalitySeries(climate_values, [months] * N, [doy] * N, [years] * N, climate_bands),
)
bench = Benchmark(
    name='synthetic',
    label_kind='multiclass',
    native=native,
    labels=np.arange(N, dtype=np.int64) % 2,
    groups=np.array(['a', 'a', 'b', 'b'], dtype=object),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype=np.float32), N, axis=0),
    label_names=['zero', 'one'],
    years=np.full(N, 2021, dtype=np.int64),
)

s2_monthly, _, _, _ = bench.monthly('s2')
s1_monthly, _, _, _ = bench.monthly('s1')
climate_monthly, _, _, _ = bench.monthly('climate')
print('s2 monthly', s2_monthly.shape, bench.s2_bands)
print('s1 monthly', s1_monthly.shape, bench.s1_bands)
print('climate monthly', climate_monthly.shape, bench.climate_bands)
print('latlon', bench.latlon.shape, 'years', bench.years.shape)

s2 monthly (4, 12, 14) ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1 monthly (4, 12, 2) ['VV', 'VH']
climate monthly (4, 12, 4) ['temperature', 'precipitation', 'elevation', 'slope']
latlon (4, 2) years (4,)


## Wrapper mapping

`_bench_to_galileo` is the conversion used internally by `encode`. It returns all grouped Galileo inputs and masks without loading model weights.

In [4]:
enc = galileo.GalileoModel(device='cpu', model_size='nano', patch_size=1)
inputs = enc._bench_to_galileo(bench)

names = ['x_st', 'sp_x', 't_x', 'st_x', 's_t_m', 'sp_m', 't_m', 'st_m', 'months']
for name, value in zip(names, inputs):
    print(name, value.shape, value.dtype)

print('embedding dim for nano:', enc.embedding_dim)

x_st (4, 1, 1, 12, 13) float32
sp_x (4, 1, 1, 16) float32
t_x (4, 12, 6) float32
st_x (4, 18) float32
s_t_m (4, 1, 1, 12, 7) float32
sp_m (4, 1, 1, 3) float32
t_m (4, 12, 3) float32
st_m (4, 4) float32
months (4, 12) int64
embedding dim for nano: 128


## Forward pass -> embedding

The real forward pass may download Galileo weights if they are not already cached. It is disabled by default so the notebook can be used as an input-shape reference first.

In [5]:
RUN_REAL_FORWARD = False

enc = galileo.GalileoModel(device='cpu', model_size='nano', patch_size=1)

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))

Set RUN_REAL_FORWARD = True to run the frozen encoder.
